In [ ]:
!pip install git+https://github.com/soilwater/ksmesopy.git

In [ ]:
import ksmesopy as ms
import matplotlib.pyplot as plt
import numpy as np

### Station metadata

In [ ]:
ms.get_stations(names_only=True)[:5]   # first 5 station names

### Download data
Min/max temperature and wind speed max are **daily-only** variables. Use `list_variables(interval='hour')` to see what is available at each interval.

In [ ]:
df_h = ms.request_data(
    station='Manhattan',
    start='2025-06-01',
    end='2025-06-07',
    interval='hour',
    variables=['TEMP2MAVG', 'RELHUM2MAVG', 'VPDEFAVG', 'SRAVG',
               'WSPD2MAVG', 'PRECIP'],
)
df_h.head(3)

In [ ]:
df_d = ms.request_data(
    station='Manhattan',
    start='2025-01-01',
    end='2025-12-31',
    interval='day',
    variables=['TEMP2MAVG', 'TEMP2MMIN', 'TEMP2MMAX',
               'RELHUM2MMIN', 'RELHUM2MMAX',
               'SRAVG', 'WSPD2MAVG', 'PRECIP'],
)
df_d.head(3)

### Meteogram (hourly)

In [ ]:
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 7))
ms.plot_temperature(axes[0], df_h, ['TEMP2MAVG'])
ms.plot_precip(axes[1], df_h, 'PRECIP')
ms.plot_humidity(axes[2], df_h, 'RELHUM2MAVG')
plt.tight_layout()
plt.show()

### Vapor pressure deficit (VPD)
`VPDEFAVG` is measured directly by the station. `vapor_pressure_deficit()` lets you compute it from temperature and relative humidity when only those are available.

In [ ]:
# Compute VPD from T and RH (hourly), then compare to measured VPDEFAVG
df_h['VPD_calc'] = ms.vapor_pressure_deficit(df_h['TEMP2MAVG'], df_h['RELHUM2MAVG'])

fig, ax = plt.subplots(figsize=(10, 3))
ms.plot_vpd(ax, df_h, variables=['VPDEFAVG', 'VPD_calc'])
ax.set_title('Measured vs. calculated VPD')
plt.tight_layout()
plt.show()

### Solar radiation vs. extraterrestrial radiation
Extraterrestrial radiation (Ra) is the theoretical maximum — the gap between Ra and observed solar radiation reflects cloud cover and atmospheric losses.

In [ ]:
# Station coordinates for Manhattan, KS
LAT, ELEV = 39.19, 325.0

df_d['DOY'] = df_d['TIMESTAMP'].dt.dayofyear
_, Ra = ms.reference_et_penman_monteith(
    df_d['DOY'], LAT, ELEV,
    df_d['TEMP2MMIN'], df_d['TEMP2MMAX'],
    df_d['SRAVG'], df_d['WSPD2MAVG'],
    rhmin=df_d['RELHUM2MMIN'], rhmax=df_d['RELHUM2MMAX'],
)
# Convert Ra from MJ/m²/day → W/m² for a fair comparison on the same axis
df_d['Ra_Wm2'] = Ra * 1e6 / 86400

fig, ax = plt.subplots(figsize=(10, 3))
ms.plot_solar_radiation(ax, df_d, variables=['SRAVG', 'Ra_Wm2'])
ax.set_title('Observed solar radiation (Rs) vs. extraterrestrial radiation (Ra)')
plt.tight_layout()
plt.show()

### Reference evapotranspiration (ETo)
Penman-Monteith (FAO-56) requires full met data; Hargreaves–Samani only needs temperature. Both return `(ETo [mm/day], Ra [MJ m⁻² day⁻¹])`.

In [ ]:
ETo_pm, _ = ms.reference_et_penman_monteith(
    df_d['DOY'], LAT, ELEV,
    df_d['TEMP2MMIN'], df_d['TEMP2MMAX'],
    df_d['SRAVG'], df_d['WSPD2MAVG'],
    rhmin=df_d['RELHUM2MMIN'], rhmax=df_d['RELHUM2MMAX'],
)
ETo_hg, _ = ms.reference_et_hargreaves(
    df_d['DOY'], LAT, df_d['TEMP2MMIN'], df_d['TEMP2MMAX'],
)
df_d['ETo_PM'] = ETo_pm
df_d['ETo_HG'] = ETo_hg

fig, ax = plt.subplots(figsize=(10, 3))
ms.plot_et(ax, df_d, variables=['ETo_PM', 'ETo_HG'])
ax.set_title('ETo — Penman-Monteith vs. Hargreaves-Samani')
plt.tight_layout()
plt.show()

### Soil moisture
Fetch Ka and EC alongside the firmware VWC values, apply the KSU site-specific calibration, then compute storage.

In [ ]:
df_soil = ms.request_data(
    station='Manhattan',
    start='2025-01-01',
    end='2025-12-31',
    interval='day',
    variables=[
        'VWC5CM',  'VWC10CM',  'VWC20CM',  'VWC50CM',
        'SOILKA5CM', 'SOILKA10CM', 'SOILKA20CM', 'SOILKA50CM',
        'SOILEC5CM', 'SOILEC10CM', 'SOILEC20CM', 'SOILEC50CM',
    ],
)
df_soil = ms.calibrate_vwc(df_soil)
df_soil = ms.compute_soil_water_storage(df_soil)
df_soil[['TIMESTAMP', 'VWC5CM', 'VWC10CM', 'VWC20CM', 'VWC50CM', 'STORAGE_MM']].head(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ms.plot_vwc(ax, df_soil)
plt.tight_layout()
plt.show()

### Optional: rename columns to snake_case

In [ ]:
ms.rename_columns(df_d).head(3)